# CAGEFusion — User Guide

This notebook demonstrates the three main workflows:

| Section | What you'll learn |
|---------|-------------------|
| [1. MoleculeNet Benchmarks](#1-moleculenet-benchmarks) | Run a standard benchmark in a single function call |
| [2. Train on Custom Data](#2-train-on-custom-data) | Train from your own CSV / DataFrames |
| [3. Inference & Explainability](#3-inference--explainability) | Predict + understand model decisions |

> **Hardware**: A CUDA GPU is recommended but not required.  All cells fall back to CPU automatically.

## Setup

```bash
pip install cage_fusion
# or from source:
# pip install -e .
```

In [ ]:
# Verify installation
import cage_fusion
print(f"cage_fusion v{cage_fusion.__version__}")

---
## 1. MoleculeNet Benchmarks

`run_moleculenet_benchmark()` handles everything end-to-end:
data download → featurisation → training → test evaluation.

Supported datasets (via DeepChem): `bace_classification`, `bbbp`,
`clintox`, `hiv`, `muv`, `sider`, `tox21`, `toxcast`, and more.

In [ ]:
from cage_fusion.benchmarks import run_moleculenet_benchmark

results = run_moleculenet_benchmark(
    dataset="bace_classification",
    output_dir="runs/bace",
    splitter="scaffold",   # reproducible chemistry-aware split
    num_epochs=50,
    seed=42,
)

print(f"Test ROC-AUC : {results['test_auc']:.4f}")
print(f"Test MCC     : {results['test_mcc']:.4f}")
print(f"Test PR-AUC  : {results['test_pr']:.4f}")

### Per-task breakdown

In [ ]:
import pandas as pd

rows = [
    {"task": task, "AUC": auc, "PR-AUC": pr, "MCC": mcc}
    for task, (mcc, auc, pr) in zip(
        results["label_names"], results["per_task_metrics"]
    )
]
pd.DataFrame(rows).set_index("task").round(4)

### Training curves

In [ ]:
import matplotlib.pyplot as plt

history = results["history"]
epochs  = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, metric in zip(axes, ["loss", "auc", "mcc"]):
    ax.plot(epochs, history[f"train_{metric}"], label="train")
    ax.plot(epochs, history[f"val_{metric}"],   label="val",   linestyle="--")
    ax.set_title(metric.upper())
    ax.set_xlabel("Epoch")
    ax.legend()

fig.suptitle("BACE training history", fontsize=13)
plt.tight_layout()
plt.show()

### Multi-task benchmark (Tox21 — 12 tasks)

In [ ]:
results_tox21 = run_moleculenet_benchmark(
    dataset="tox21",
    output_dir="runs/tox21",
    num_epochs=30,
)
print(f"Tox21 macro-avg AUC: {results_tox21['test_auc']:.4f}")

---
## 2. Train on Custom Data

The workflow mirrors HuggingFace Transformers:

1. **`CageFusionDataModule`** — featurise your data into ready-to-use DataLoaders
2. **`CageFusionConfig`** — define architecture & task metadata
3. **`AutoCageFusion`** — instantiate the right task head automatically
4. **`TrainingArguments` + `Trainer`** — train with a single `.train()` call

### 2a. Prepare a toy dataset

In [ ]:
import pandas as pd
from rdkit import Chem

# Tiny demo dataset — replace with your own CSV
smiles_list = [
    "CC(=O)Oc1ccccc1C(=O)O",   # aspirin        — active=1
    "c1ccc2c(c1)cc1ccc3cccc4ccc2c1c34",  # pyrene — active=1
    "CC(C)Cc1ccc(cc1)C(C)C(=O)O",  # ibuprofen  — active=0
    "OC(=O)c1ccccc1O",          # salicylic acid — active=0
    "CC12CCC3C(C1CCC2O)CCC4=CC(=O)CCC34C",  # testosterone — active=1
    "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",  # caffeine   — active=0
    "c1ccc2cc3ccccc3cc2c1",     # anthracene     — active=1
    "O=C(O)c1ccc(cc1)N",        # 4-aminobenzoic — active=0
] * 20  # repeat to get a larger dataset for demo

labels = [1, 1, 0, 0, 1, 0, 1, 0] * 20

df = pd.DataFrame({"SMILES": smiles_list, "active": labels})
df.to_csv("/tmp/demo_compounds.csv", index=False)
print(f"{len(df)} molecules, {df['active'].mean():.0%} positive")

### 2b. Featurise with `CageFusionDataModule`

In [ ]:
from cage_fusion.data import CageFusionDataModule

dm = CageFusionDataModule.from_csv(
    "/tmp/demo_compounds.csv",
    label_cols=["active"],
    model_checkpoint="DeepChem/ChemBERTa-77M-MTR",  # HF sequence encoder
    val_split=0.15,
    test_split=0.10,
    batch_size=32,
    cache_dir="/tmp/demo_features",
)

print(dm)  # CageFusionDataModule(labels=['active'], train=..., val=..., test=...)

> **Tip:** For large datasets or multi-GPU setups, increase `num_workers` and set `cache_dir` to a fast SSD.

### 2c. Configure the model

In [ ]:
from cage_fusion import CageFusionConfig

config = CageFusionConfig(
    # Task definition
    num_labels=len(dm.label_names),
    model_task="classification",   # or "regression" for ADMET values
    label_names=dm.label_names,

    # Architecture (defaults work well — tweak for your domain)
    attn_mode="self_graph",  # 'cross' | 'self_tokens' | 'self_graph' | 'self_both'
    use_fg_prompt=True,      # functional-group chemical prompting
    use_aux_features=True,   # 217 RDKit physicochemical descriptors
)
print(config)

### 2d. Instantiate and train

In [ ]:
from cage_fusion import AutoCageFusion
from cage_fusion.training import Trainer, TrainingArguments

# AutoCageFusion picks CAGEFusionForMultiLabelClassification or
# CAGEFusionForRegression based on config.model_task
model = AutoCageFusion.from_config(config)

args = TrainingArguments(
    output_dir="runs/demo",
    num_epochs=10,
    learning_rate=3e-4,
    batch_size=32,
)

# No optimizer needed — Trainer builds Adam automatically
trainer = Trainer(
    model=model,
    args=args,
    train_loader=dm.train_loader,
    val_loader=dm.val_loader,
)

history = trainer.train()
print(f"Best val AUC: {max(history['val_auc']):.4f}")

### 2e. Phased / staged fine-tuning (advanced)

For larger datasets, use `staged_finetune()` which runs a 4-phase
curriculum: warmup → core encoder → aux-features warmup → full unfreeze.

In [ ]:
# trainer.staged_finetune(
#     num_epochs_warmup=5,
#     num_epochs_phase1=15,
#     num_epochs_aux_warmup=5,
#     num_epochs_phase2=25,
# )
print("(Cell commented out — run it for large datasets.)")

### 2f. Save the scaler alongside the model

In [ ]:
# The Trainer already writes best_model.pt to args.checkpoints_dir.
# Save the aux-features scaler so the pipeline can load it:
dm.save_scaler(args.checkpoints_dir)
print("Scaler saved to", args.checkpoints_dir)

### 2g. Regression example (ADMET values)

In [ ]:
import numpy as np

# Synthetic ADMET dataset
admet_df = pd.DataFrame({
    "SMILES": smiles_list,
    "logP":   np.random.uniform(-2, 6, len(smiles_list)),
    "solubility": np.random.uniform(-6, 1, len(smiles_list)),
})
admet_df.to_csv("/tmp/demo_admet.csv", index=False)

# -----------------------------
# dm_admet = CageFusionDataModule.from_csv(
#     "/tmp/demo_admet.csv",
#     label_cols=["logP", "solubility"],
#     model_checkpoint="DeepChem/ChemBERTa-77M-MTR",
# )
#
# config_reg = CageFusionConfig(
#     num_labels=2,
#     model_task="regression",
#     label_names=["logP", "solubility"],
# )
# model_reg = AutoCageFusion.from_config(config_reg)
# trainer_reg = Trainer(model_reg, args, dm_admet.train_loader, dm_admet.val_loader)
# trainer_reg.train()
# -----------------------------
print("Regression workflow ready — uncomment the cells above.")

---
## 3. Inference & Explainability

### 3a. Load a pipeline

In [ ]:
from cage_fusion import CageFusionPipeline

# Point to the directory that contains best_model.pt + aux_features_scaler.pkl
pipe = CageFusionPipeline.from_pretrained("runs/demo/checkpoints")

### 3b. Single SMILES → dict

In [ ]:
aspirin = "CC(=O)Oc1ccccc1C(=O)O"
result  = pipe(aspirin)

print(result)

### 3c. List of SMILES → list of dicts

In [ ]:
compounds = [
    "CC(=O)Oc1ccccc1C(=O)O",           # aspirin
    "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",   # caffeine
    "c1ccc2cc3ccccc3cc2c1",            # anthracene
]

results = pipe(compounds)
pd.DataFrame(results)

### 3d. DataFrame → DataFrame (batch inference)

In [ ]:
test_df = pd.read_csv("/tmp/demo_compounds.csv")
predictions = pipe(test_df)
predictions.head()

### 3e. Gradient saliency — which tokens matter?

In [ ]:
from cage_fusion.inference import GradientExplainer

# Reuse the loaded pipeline (no model reload)
explainer = GradientExplainer(pipe)

explanation = explainer.explain(
    aspirin,
    target_task="active",  # one of dm.label_names
)

print(f"Prediction : {explanation['probability']:.3f}  "
      f"(class {explanation['predicted_class']})")
print(f"Threshold  : {explanation['threshold']:.3f}")

### 3f. Visualise token saliency

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

tokens   = explanation["tokens"]
saliency = explanation["token_saliency"]

# Normalise to [0, 1] for colour intensity
sal_norm = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)

fig, ax = plt.subplots(figsize=(max(10, len(tokens) * 0.6), 1.8))
for i, (tok, s) in enumerate(zip(tokens, sal_norm)):
    ax.barh(0, 1, left=i, color=plt.cm.Reds(0.2 + 0.8 * s), edgecolor="white")
    ax.text(i + 0.5, 0, tok, ha="center", va="center", fontsize=9, color="black")

ax.set_xlim(0, len(tokens))
ax.set_yticks([])
ax.set_title(f"Token saliency — '{explanation['task']}'  "
             f"p={explanation['probability']:.3f}")
plt.tight_layout()
plt.show()

### 3g. Auxiliary feature saliency (top RDKit descriptors)

In [ ]:
aux_sal   = np.abs(explanation["aux_saliency"])
top_k     = 10
top_idxs  = np.argsort(aux_sal)[-top_k:][::-1]

fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(range(top_k), aux_sal[top_idxs], color="steelblue")
ax.set_yticks(range(top_k))
ax.set_yticklabels([f"feature_{i}" for i in top_idxs])
ax.set_xlabel("Gradient magnitude")
ax.set_title("Top-10 most influential RDKit descriptors")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### 3h. Attention visualisation (requires RDKit SVG renderer)

In [ ]:
import os, tempfile
from IPython.display import Image, display

attn_dir = tempfile.mkdtemp()

# Re-run inference with attention plotting enabled
pipe.predict(
    pd.DataFrame([{"SMILES": aspirin}]),
    plot_all_attention=True,
    attn_plot_dir=attn_dir,
)

# Display the per-atom contribution map
img_path = os.path.join(attn_dir, "idx_0", "atom_total_contrib.png")
if os.path.exists(img_path):
    display(Image(filename=img_path))
else:
    print("Attention image not found — check RDKit installation.")

---
## Summary

```python
# ── MoleculeNet benchmark (1 call) ─────────────────────────────────────────
from cage_fusion.benchmarks import run_moleculenet_benchmark
results = run_moleculenet_benchmark("bace_classification", num_epochs=50)

# ── Custom data training ────────────────────────────────────────────────────
from cage_fusion.data     import CageFusionDataModule
from cage_fusion          import CageFusionConfig, AutoCageFusion
from cage_fusion.training import Trainer, TrainingArguments

dm      = CageFusionDataModule.from_csv("compounds.csv", label_cols=["active"])
config  = CageFusionConfig(num_labels=1, model_task="classification",
                            label_names=["active"])
model   = AutoCageFusion.from_config(config)
trainer = Trainer(model, TrainingArguments(output_dir="runs/my_model"),
                  dm.train_loader, dm.val_loader)
trainer.train()
dm.save_scaler("runs/my_model/checkpoints")

# ── Inference ───────────────────────────────────────────────────────────────
from cage_fusion import CageFusionPipeline
pipe = CageFusionPipeline.from_pretrained("runs/my_model/checkpoints")
pipe("CC(=O)Oc1ccccc1C(=O)O")          # -> dict
pipe(["SMILES1", "SMILES2"])            # -> list[dict]

# ── Explainability ──────────────────────────────────────────────────────────
from cage_fusion.inference import GradientExplainer
exp = GradientExplainer(pipe)
out = exp.explain("CC(=O)Oc1ccccc1C(=O)O", target_task="active")
# out["tokens"], out["token_saliency"], out["aux_saliency"]
```